# Setup

In [1]:
import pandas as pd
import pyBigWig
import numpy as np

## Helper functions

In [2]:
def load_regions(window_files, buffer=100, block_size=300):
    # Read and combine the Skipper reproducible-window tables.
    regions = pd.concat([pd.read_csv(f, sep="\t") for f in window_files], ignore_index=True)

    # Add buffer around each region.
    regions["start_buffered"] = (regions["start"] - buffer).clip(lower=0)
    regions["end_buffered"] = regions["end"] + buffer

    # Sort so overlapping windows can be merged separately on each strand.
    regions = regions.sort_values(["chr", "strand", "start_buffered", "end_buffered"])

    merged = []

    for (chrom, strand), group in regions.groupby(["chr", "strand"], sort=False):
        current_start = None
        current_end = None

        for row in group.itertuples(index=False):
            if current_start is None:
                current_start = row.start_buffered
                current_end = row.end_buffered
            elif row.start_buffered < current_end:
                current_end = max(current_end, row.end_buffered)
            else:
                merged.append({"chr": chrom, "strand": strand,
                               "start_buffered": current_start, "end_buffered": current_end})
                current_start = row.start_buffered
                current_end = row.end_buffered

        merged.append({"chr": chrom, "strand": strand,
                       "start_buffered": current_start, "end_buffered": current_end})

    # Convert merged regions into fixed-length blocks.
    blocks = []

    for region_id, region in enumerate(merged):
        chrom = region["chr"]
        strand = region["strand"]
        start = region["start_buffered"]
        end = region["end_buffered"]
        length = end - start

        if length < block_size:
            extra = block_size - length
            start = max(0, start - extra // 2)
            end = start + block_size

        block_start = start
        block_number = 0

        while block_start + block_size <= end:
            blocks.append({"chr": chrom, "strand": strand,
                           "start_buffered": block_start, "end_buffered": block_start + block_size,
                           "region_id": region_id, "block_number": block_number})
            block_start += block_size
            block_number += 1

        if block_start < end:
            final_start = end - block_size
            blocks.append({"chr": chrom, "strand": strand,
                           "start_buffered": final_start, "end_buffered": end,
                           "region_id": region_id, "block_number": block_number})

    return(pd.DataFrame(blocks))

def extract_bigwig_signal(plus_bigwigs, minus_bigwigs, regions):
    plus_bws = [pyBigWig.open(f) for f in plus_bigwigs]
    minus_bws = [pyBigWig.open(f) for f in minus_bigwigs]

    extracted = []

    for row in regions.itertuples(index=False):
        # Choose the strand-matched replicate bigWigs.
        if row.strand == "+":
            bws = plus_bws
        elif row.strand == "-":
            bws = minus_bws
        else:
            continue

        # Skip chromosomes not present in all replicates.
        if not all(row.chr in bw.chroms() for bw in bws):
            continue

        chrom_size = min(bw.chroms(row.chr) for bw in bws)
        start = max(0, row.start_buffered)
        end = min(row.end_buffered, chrom_size)

        # Extract and clean each replicate.
        replicate_values = []
        for bw in bws:
            values = bw.values(row.chr, start, end, numpy=True)
            values = np.nan_to_num(values, nan=0.0)
            replicate_values.append(values)

        # Average signal across replicates.
        values = np.mean(replicate_values, axis=0)

        # Orient all windows in the 5' -> 3' direction.
        if row.strand == "-":
            values = values[::-1]

    extracted.append({
        "chr": row.chr,
        "start": start,
        "end": end,
        "strand": row.strand,
        "region_id": row.region_id,
        "block_number": row.block_number,
        "signal": values
    })
    
    for bw in plus_bws + minus_bws:
        bw.close()

    return(extracted)

def save_pair_dataset(signal_A, signal_B, output_file, rbp_A, rbp_B, cell_type):
    # Create the model-ready N x 2 x 300 signal array.
    signals = np.stack([
        np.stack([a["signal"], b["signal"]])
        for a, b in zip(signal_A, signal_B)
    ]).astype(np.float32)

    np.savez_compressed(
        output_file,
        signals=signals,
        chrom=np.array([x["chr"] for x in signal_A]),
        start=np.array([x["start"] for x in signal_A]),
        end=np.array([x["end"] for x in signal_A]),
        strand=np.array([x["strand"] for x in signal_A]),
        region_id=np.array([x["region_id"] for x in signal_A]),
        block_number=np.array([x["block_number"] for x in signal_A]),
        rbp_A=np.array(rbp_A),
        rbp_B=np.array(rbp_B),
        cell_type=np.array(cell_type)
    )

# Run basic extraction

In [3]:
regions = load_regions([
    "~/projects/Qingfei_skipper_run/output/reproducible_enriched_windows/ADARp110.reproducible_enriched_windows.tsv.gz",
    "~/projects/Qingfei_skipper_run/output/reproducible_enriched_windows/ADARp150.reproducible_enriched_windows.tsv.gz"
], buffer=100)

signal_A = extract_bigwig_signal(
    [
        "/tscc/nfs/home/kflanagan/projects/Qingfei_skipper_run/output/secondary_results/bigwigs/scaled/plus/ADARp110_IP_1.scaled.plus.bw",
        "/tscc/nfs/home/kflanagan/projects/Qingfei_skipper_run/output/secondary_results/bigwigs/scaled/plus/ADARp110_IP_2.scaled.plus.bw"
    ],
    [
        "/tscc/nfs/home/kflanagan/projects/Qingfei_skipper_run/output/secondary_results/bigwigs/scaled/minus/ADARp110_IP_1.scaled.minus.bw",
        "/tscc/nfs/home/kflanagan/projects/Qingfei_skipper_run/output/secondary_results/bigwigs/scaled/minus/ADARp110_IP_2.scaled.minus.bw"
    ],
    regions
)

signal_B = extract_bigwig_signal(
    [
        "/tscc/nfs/home/kflanagan/projects/Qingfei_skipper_run/output/secondary_results/bigwigs/scaled/plus/ADARp150_IP_1.scaled.plus.bw",
        "/tscc/nfs/home/kflanagan/projects/Qingfei_skipper_run/output/secondary_results/bigwigs/scaled/plus/ADARp150_IP_2.scaled.plus.bw"
    ],
    [
        "/tscc/nfs/home/kflanagan/projects/Qingfei_skipper_run/output/secondary_results/bigwigs/scaled/minus/ADARp150_IP_1.scaled.minus.bw",
        "/tscc/nfs/home/kflanagan/projects/Qingfei_skipper_run/output/secondary_results/bigwigs/scaled/minus/ADARp150_IP_2.scaled.minus.bw"
    ],
    regions
)

In [4]:
save_pair_dataset(signal_A, signal_B, "/tscc/nfs/home/kflanagan/scratch/RNA_protein_grammar/ADARp110_ADARp150.npz",
                  rbp_A="ADARp110", rbp_B="ADARp150", cell_type="HepG2")